# 📓 실습 02. 규칙 기반 & LLM 지식 트리플렛(Triplet) 추출

이 실습에서는 파싱된 마크다운 청크에서 반도체 CMP 설비의 개체명(Entity)과 관계(Relationship) 삼항조를 정량적으로 추출하는 파이프라인을 구축합니다. API 키가 없어도 원활하게 동작하는 규칙 기반 파서(Regex Rule Extractor)도 제공됩니다.

In [ ]:
import os
import json
import re

# 1. 파싱된 청크 데이터 로드
chunks_path = os.path.join("..", "01_structural_ingest_parsing", "data", "parsed_chunks.json")
with open(chunks_path, "r", encoding="utf-8") as f:
    chunks = json.load(f)
print(f"불러온 총 청크 수: {len(chunks)} 개")

In [ ]:
# 2. 규칙 기반(Regex Rule-based) 트리플렛 추출기 구현
# (API 키 미연동 시의 예외 처리 로직)
def extract_triplets_rule_based(chunks):
    triplets = []
    # 대표적인 CMP 설비 개체 사전 정의
    entities = ["CMP 장비", "폴리싱 플래튼", "폴리싱 패드", "플래튼 구동 모터",
                "폴리싱 헤드", "리테이너 링", "헤드 멤브레인", "슬러리 공급 암",
                "슬러리 토출 노즐", "유량 제어 밸브", "패드 컨디셔너", "다이아몬드 컨디셔닝 디스크"]
    
    relations = [
        ("구성", ["구성", "포함", "부착", "연결"]), 
        ("유발", ["유발", "원인", "발생", "초래"]), 
        ("조치", ["조치", "SOP", "해결", "SOP-01", "SOP-02", "SOP-03"])
    ]
    
    for chunk_idx, chunk in enumerate(chunks):
        content = chunk["content"]
        # 본문에 포함된 엔티티들 확인
        found_entities = [ent for ent in entities if ent in content]
        
        # 기하학적 매칭 (동일 문장 내에 있을 때 트리플렛 생성)
        sentences = re.split(r'[.\n]', content)
        for sentence in sentences:
            ents_in_sent = [ent for ent in found_entities if ent in sentence]
            if len(ents_in_sent) >= 2:
                # 문장에서 관계성 단어 확인
                assigned_relation = "RELATED_TO"
                for rel_type, keywords in relations:
                    if any(kw in sentence for kw in keywords):
                        assigned_relation = rel_type.upper()
                        break
                
                # 트리플렛 구성
                triplets.append({
                    "subject": ents_in_sent[0],
                    "relation": assigned_relation,
                    "object": ents_in_sent[1],
                    "chunk_id": chunk_idx,
                    "context": sentence.strip()
                })
                
    # 중복 제거
    unique_triplets = []
    seen = set()
    for trip in triplets:
        key = (trip["subject"], trip["relation"], trip["object"])
        if key not in seen:
            seen.add(key)
            unique_triplets.append(trip)
            
    return unique_triplets

triplets = extract_triplets_rule_based(chunks)
print(f"추출된 총 지식 트리플렛 수: {len(triplets)} 개\n")

# 샘플 트리플렛 출력
print("=== [추출 트리플렛 샘플] ===")
for t in triplets[:3]:
    print(f"({t['subject']}) - [{t['relation']}] -> ({t['object']}) [근거: {t['context']}]")

In [ ]:
# 3. 추출된 트리플렛 저장
os.makedirs("data", exist_ok=True)
output_path = os.path.join("data", "extracted_triplets.json")
with open(output_path, "w", encoding="utf-8") as f:
    json.dump(triplets, f, indent=2, ensure_ascii=False)
print(f"트리플렛 저장 완료: {output_path}")